# De Politieke Mens: Nevenfuncties & Financiën

**Thema:** Extra activiteiten en inkomstenbronnen van Kamerleden.

**Kernvragen:** Hoeveel nevenfuncties hebben politici gemiddeld, in welke sectoren liggen deze, en wat zijn de gerapporteerde inkomsten per jaar en soort vergoeding?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_nevenfunctie` en `persoon_nevenfunctie_inkomsten`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

# Algemene informatie 

## Verschillende partij in de deze database

In [ ]:
verschillende_partijen = query_df("SELECT DISTINCT naam_nl FROM silver.fractie ORDER BY naam_nl")
a= len(verschillende_partijen)
print("Aantal verschillende rijen: ", a)
verschillende_partijen


## Aantal politici 

In [ ]:
aantal_politici= query_df("SELECT distinct voornamen,achternaam FROM silver.persoon ")
print("Aantal politici in dit bestand", len(aantal_politici))
aantal_politici

## aantal verschillende omschrijvingen

In [ ]:
aantal_omschrijvingen = query_df("""
SELECT omschrijving, COUNT(*) AS aantal
FROM silver.persoon_nevenfunctie
GROUP BY omschrijving
ORDER BY aantal DESC
""")
print("Aantal verschillende omschrijvingen: ",len(aantal_omschrijvingen))
aantal_omschrijvingen.head(50)

## Onderwerp 1 Gemiddeld aantal nevenfunctie 

Geef hier een beschrijving van het onderwerp of deelvraag die je wilt beantwoorden.

In [ ]:
gemiddelde_zonder_0 = query_df("""
WITH aantal_zonder AS (
SELECT persoon_id, count(*) as aantal_per 
FROM silver.persoon_nevenfunctie n
GROUP BY persoon_id
)
SELECT AVG(aantal_per) AS gem_nevenfunctie_persoon_z , 
MAX(aantal_per) AS maxium_nevenfunctie_persoon_z, 
MIN(aantal_per) AS minimum_nevenfuntie_persoon_z
FROM aantal_zonder 
""")
gemiddelde_zonder_0

In [ ]:
gemiddelde_met_0 = query_df("""  
WITH aantal_met AS (
    SELECT p.id, count(n.persoon_id) as aantal_per 
    FROM silver.persoon p 
    LEFT JOIN silver.persoon_nevenfunctie n on n.persoon_id = p.id
    GROUP BY p.id
    )
SELECT
   AVG(aantal_per) AS gem_nevenfunctie_persoon_m,
   MAX(aantal_per) as maximum_nevenfuncties_persoon_m,
   MIN(aantal_per) as minimum_nevenfuncties_persoon_m
  FROM aantal_met
""")
gemiddelde_met_0

In [ ]:
maxium_0 = query_df("""  
WITH aantal AS (
    SELECT persoon_id, count(*) as aantal_neuven_functie 
    FROM silver.persoon_nevenfunctie n 
    GROUP BY persoon_id
)

SELECT voornamen,achternaam,aantal_neuven_functie 
FROM aantal a
LEFT JOIN silver.persoon p on a.persoon_id = p.id
WHERE aantal_neuven_functie  = (SELECT MAX(aantal_neuven_functie) FROM aantal)
GROUP BY aantal_neuven_functie , voornamen,achternaam
""")
maxium_0

## Onderwerp 2 VERSCHILLENDE SECTOREN 
  

In [ ]:
aantal_omschrijvingen= query_df("""select  count(distinct omschrijving) as aantal_verschillende
from silver.persoon_nevenfunctie
""")
print(aantal_omschrijvingen)


Het was niet mogelijk om het via de omschrijving te doen, omdat er 1814 verschillende omschrijvingen zijn. Dus heb ik zelf sectoren gemaakt.

In [ ]:
sector_nevenfunctie = query_df("""
with sectoren as (
select
    id, omschrijving,
    case
        when lower(omschrijving) like '%ziekenhuis%'
          or lower(omschrijving) like '%zorg%'
          or lower(omschrijving) like '%gezondheid%' then 'Zorg'
        when lower(omschrijving) like '%onderwijs%'
          or lower(omschrijving) like '%school%'
          or lower(omschrijving) like '%universiteit%'
          or lower(omschrijving) like '%hogeschool%' then 'Onderwijs'
        when lower(omschrijving) like '%bank%'
          or lower(omschrijving) like '%verzekering%'
          or lower(omschrijving) like '%financ%'
          or lower(omschrijving) like '%management%' then 'Financieel'
        when lower(omschrijving) like '%gemeente%'
          or lower(omschrijving) like '%provincie%'
          or lower(omschrijving) like '%waterschap%'
          or lower(omschrijving) like '%kamer%'
          or lower(omschrijving) like '%overheid%' then 'Overheid'
        when lower(omschrijving) like '%stichting%'
          or lower(omschrijving) like '%vereniging%'
          or lower(omschrijving) like '%fonds%'
          or lower(omschrijving) like '%ambassadeur%'
          or lower(omschrijving) like '%vrijwillig%'then 'Maatschappelijk'
        else 'Overig'
    end as sector
from silver.persoon_nevenfunctie)

select sector ,count(*) as aantal
from sectoren
group by sector

""")

trefwoorden
# select sector ,count(*)
#from sectoren
#group by sector


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    sector_nevenfunctie["aantal"],
    labels=sector_nevenfunctie["sector"],
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)
ax.set_title("Nevenfuncties per sector")
plt.tight_layout()
plt.show()

Deze opbouw is niet handig omdat bijvoorbeeld bank kan fincieel zijn, maar maatschappelijk door de voedselbank. 

In [ ]:
overig = query_df("""
SELECT omschrijving, COUNT(*) AS aantal
FROM silver.persoon_nevenfunctie
GROUP BY omschrijving
ORDER BY aantal DESC
""")
overig.head(50)

In [ ]:
maxium = query_df("""  
WITH aantal AS (
    SELECT persoon_id, count(*) as aantal_neuven_functie 
    FROM silver.persoon_nevenfunctie n 
    GROUP BY persoon_id
)

SELECT voornamen,achternaam,aantal_neuven_functie 
FROM aantal a
LEFT JOIN silver.persoon p on a.persoon_id = p.id
WHERE aantal_neuven_functie  = (SELECT MAX(aantal_neuven_functie) FROM aantal)
GROUP BY aantal_neuven_functie , voornamen,achternaam
""")
maxium

De persoon met de meeste nevenfunctie is Alexander Pechtold. Deze cijfers kan verkeerd beeld geven, omdat hij lang in de politiek zit en is dus van meederre jaren. Dit betekent niet dat hij per jaar de meeste nevenfunctie heeft of het meeste verdiend

In [ ]:
partijen = query_df("""
select Distinct p.id,voornamen,achternaam,naam_nl,
from silver.toezegging_toegezegd_aan t
join silver.persoon p on p.id = t.persoon_id
join silver.fractie f on f.id = t.fractie_id

""")
partijen

In [ ]:
partijen = query_df("""
select voornamen,achternaam,naam_nl,zp.tot_en_met	
from silver.fractie_zetel_persoon zp
join silver.persoon p on p.id = zp.persoon_id
join silver.fractie_zetel z on z.id= zp.fractie_zetel_id
join silver.fractie f on f.id = z.fractie_id

""")
#where f.naam_nl in (select voornamen,achternaam from  silver.fractie_zetel_persoon zp
partijen

In [ ]:
overstappen = query_df("""
WITH lidmaatschap AS (
    SELECT
        zp.persoon_id,
        f.id AS fractie_id,
        f.naam_nl AS partij,
        zp.van,
        zp.tot_en_met
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
),
volgorde AS (
    SELECT
        *,
        LEAD(partij)    OVER (PARTITION BY persoon_id ORDER BY van) AS volgende_partij,
        LEAD(fractie_id) OVER (PARTITION BY persoon_id ORDER BY van) AS volgende_fractie_id,
        LEAD(van)       OVER (PARTITION BY persoon_id ORDER BY van) AS start_volgende
    FROM lidmaatschap
)
SELECT
    pe.voornamen,
    pe.achternaam,
    v.partij        AS van_partij,
    v.tot_en_met    AS einde_vorige,
    v.volgende_partij AS naar_partij,
    v.start_volgende
FROM volgorde v
JOIN silver.persoon pe ON pe.id = v.persoon_id
WHERE v.volgende_fractie_id IS NOT NULL
  AND v.volgende_fractie_id <> v.fractie_id
  AND v.partij = 'Lijst Pim Fortuyn'
ORDER BY v.start_volgende, pe.achternaam
""")
overstappen

In [ ]:
display(query_df("DESCRIBE silver.fractie_zetel_persoon"))
display(query_df("DESCRIBE silver.fractie_zetel"))

## Onderwerp 3  Wat zijn de gerapporteerde inkomsten per jaar en soort vergoeding?

In [ ]:
vergoeding_perjaar = query_df("""
with sectoren as (
    select 
        sum(i.bedrag) as som,voornamen,achternaam,p.id,i.jaar
    from silver.persoon_nevenfunctie_inkomsten i 
    inner join silver.persoon_nevenfunctie n on n.id = i.persoon_nevenfunctie_id
    inner join silver.persoon p on p.id = n.persoon_id
    where frequentie = 'Per jaar'
    group by voornamen,achternaam,i.jaar,p.id
    order by i.jaar, sum(bedrag)
    )
select voornamen,achternaam,avg(som) as gemiddelde_bedrag_per_jaar,count(som) as aantal_jaren_nevenfuncite,sum(som)
from sectoren
group by voornamen,achternaam
order by gemiddelde_bedrag_per_jaar desc
""")
vergoeding_perjaar

In [ ]:
vergoeding_perjaar = query_df("""
with vergoeding as (select 
    sum(bedrag) as totaal_bedrag ,jaar,voornamen,achternaam
from silver.persoon_nevenfunctie_inkomsten i 
inner join silver.persoon_nevenfunctie n on n.id = i.persoon_nevenfunctie_id
inner join silver.persoon p on p.id = n.persoon_id
where i.bedrag is not null
group by voornamen,achternaam,jaar
order by sum(bedrag) desc)
select voornamen,achternaam,sum(totaal_bedrag) as totaal ,avg(totaal_bedrag) as gemiddelde ,count(jaar) as aantal_jaar
from vergoeding
group by voornamen,achternaam
order by sum(totaal_bedrag) desc
""")
vergoeding_perjaar

In [ ]:
naam= query_df("Select * FROM silver.persoon_nevenfunctie_inkomsten ")
#naam.head(50)

In [ ]:
naam= query_df("Select * FROM silver.persoon_nevenfunctie ")
#naam.head(50)

In [ ]:
inkomsten_per_jaar = query_df("""
SELECT
    p.id AS persoon_id,
    p.voornamen,
    p.achternaam,
    i.jaar,
    i.bedrag_valuta,
    COUNT(*) AS aantal_meldingen,
    SUM(i.bedrag) AS totaal_bedrag
FROM silver.persoon_nevenfunctie_inkomsten i
JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
JOIN silver.persoon p ON p.id = n.persoon_id
GROUP BY p.id, p.voornamen, p.achternaam, i.jaar, i.bedrag_valuta
ORDER BY p.achternaam, i.jaar
""")
inkomsten_per_jaar.head(50)

In [ ]:
from matplotlib.ticker import FuncFormatter

totaal_per_jaar = query_df("""
SELECT
    i.jaar,
    SUM(
        CASE
            WHEN LOWER(i.frequentie) LIKE '%maand%' THEN i.bedrag * 12
            ELSE i.bedrag
        END
    ) AS totaal_bedrag
FROM silver.persoon_nevenfunctie_inkomsten i
WHERE i.bedrag > 0
  AND i.bedrag_valuta = 'EUR'
GROUP BY i.jaar
ORDER BY i.jaar
""")
print(totaal_per_jaar)
ax = totaal_per_jaar.plot(
    x="jaar", y="totaal_bedrag", kind="bar", figsize=(10, 5), legend=False
)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.set_xlabel("Jaar")
ax.set_ylabel("Totaal gemeld bedrag (EUR)")
ax.set_title("Totale inkomsten uit nevenfuncties per jaar")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
query_df("""
SELECT bedrag_soort, frequentie, bedrag_valuta, bedrag_voorvoegsel, bedrag_achtervoegsel, COUNT(*) AS aantal
FROM silver.persoon_nevenfunctie_inkomsten
GROUP BY ALL
ORDER BY aantal DESC
""")

In [ ]:
query_df("""
SELECT persoon_nevenfunctie_id, jaar, COUNT(*) AS aantal
FROM silver.persoon_nevenfunctie_inkomsten
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY aantal DESC
""")

In [ ]:
inkomsten_per_jaar.pivot_table(
    index=["achternaam", "voornamen"],
    columns="jaar",
    values="totaal_bedrag",
    aggfunc="sum"
)

In [ ]:
totaal_per_politicus = query_df("""
WITH per_jaar AS (
    SELECT
        p.id AS persoon_id,
        p.voornamen,
        p.achternaam,
        i.jaar,
        SUM(i.bedrag) AS bedrag_jaar
    FROM silver.persoon_nevenfunctie_inkomsten i
    JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
    JOIN silver.persoon p ON p.id = n.persoon_id
    WHERE i.bedrag_valuta = 'EUR'
    GROUP BY p.id, p.voornamen, p.achternaam, i.jaar
)
SELECT
    persoon_id,
    voornamen,
    achternaam,
    COUNT(DISTINCT jaar) AS aantal_jaren,
    SUM(bedrag_jaar) AS totaal_bedrag
FROM per_jaar
GROUP BY persoon_id, voornamen, achternaam
ORDER BY totaal_bedrag DESC
""")
totaal_per_politicus


In [ ]:
per_jaar = query_df("""
SELECT
    p.id AS persoon_id,
    p.voornamen,
    p.achternaam,
    i.jaar,
    SUM(
        CASE
            WHEN LOWER(i.frequentie) LIKE '%maand%' THEN i.bedrag * 12
            ELSE i.bedrag
        END
    ) AS bedrag_jaar
FROM silver.persoon_nevenfunctie_inkomsten i
JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
JOIN silver.persoon p ON p.id = n.persoon_id

Where i.bedrag_valuta = 'EUR'
GROUP BY p.id, p.voornamen, p.achternaam, i.jaar
""")

tabel = per_jaar.pivot_table(
    index=["persoon_id", "voornamen", "achternaam"],
    columns="jaar",
    values="bedrag_jaar",
    aggfunc="sum",
    fill_value=0
)
tabel["totaal"] = tabel.sum(axis=1)
tabel = tabel.sort_values("totaal", ascending=False).reset_index()
tabel[["voornamen", "achternaam", "totaal"]].head(100)

In [ ]:
partij_overzicht = query_df("""
WITH verdieners AS (
    SELECT DISTINCT n.persoon_id
    FROM silver.persoon_nevenfunctie_inkomsten i
    JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
    WHERE i.bedrag > 0
),
politici AS (
    SELECT DISTINCT zp.persoon_id, f.naam_nl AS partij
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT
    p.partij,
    COUNT(*) AS aantal_politici,
    COUNT(v.persoon_id) AS aantal_met_inkomsten,
    ROUND(100.0 * COUNT(v.persoon_id) / COUNT(*), 1) AS percentage
FROM politici p
LEFT JOIN verdieners v ON v.persoon_id = p.persoon_id
GROUP BY p.partij
ORDER BY aantal_met_inkomsten DESC
""")
partij_overzicht

In [ ]:
N= 8
data = (
    partij_overzicht[partij_overzicht["aantal_met_inkomsten"] > 0]
    .sort_values("aantal_met_inkomsten", ascending=False)
)

top = data.head(N)
rest = data["aantal_met_inkomsten"].iloc[N:].sum()

waarden = list(top["aantal_met_inkomsten"])
labels = list(top["partij"])
if rest > 0:
    waarden.append(rest)
    labels.append("Overige partijen")

fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(
    waarden,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
    pctdistance=0.8,
)
ax.set_title("Politici met gemelde inkomsten uit nevenfuncties, per partij")
plt.tight_layout()
plt.show()


In [ ]:
partij_overzicht = query_df("""
WITH inkomsten AS (
    SELECT
        n.persoon_id,
        SUM(
            CASE
                WHEN LOWER(i.frequentie) LIKE '%maand%' THEN i.bedrag * 12
                ELSE i.bedrag
            END
        ) AS totaal_bedrag
    FROM silver.persoon_nevenfunctie_inkomsten i
    JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
    WHERE i.bedrag > 0
      AND i.bedrag_valuta = 'EUR'
    GROUP BY n.persoon_id
),
politici AS (
    SELECT DISTINCT zp.persoon_id, f.naam_nl AS partij
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT
    p.partij,
    COUNT(*) AS aantal_politici,
    COUNT(i.persoon_id) AS aantal_met_inkomsten,
    ROUND(100.0 * COUNT(i.persoon_id) / COUNT(*), 1) AS percentage,
    COALESCE(SUM(i.totaal_bedrag), 0) AS totaal_inkomsten,
    ROUND(SUM(i.totaal_bedrag) / NULLIF(COUNT(i.persoon_id), 0), 0) AS gemiddeld_per_verdiener
FROM politici p
LEFT JOIN inkomsten i ON i.persoon_id = p.persoon_id
GROUP BY p.partij
ORDER BY totaal_inkomsten DESC
""")
partij_overzicht

In [ ]:
ax = partij_overzicht.set_index("partij")["totaal_inkomsten"].plot(kind="bar", figsize=(12, 5))
ax.set_ylabel("Totaal gemeld bedrag (EUR)")
ax.set_title("Totale inkomsten uit nevenfuncties per partij")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

ax = partij_overzicht.set_index("partij")["totaal_inkomsten"].head(15).plot(kind="bar", figsize=(12, 5))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.set_ylabel("Totaal gemeld bedrag (EUR)")
ax.set_title("Totale inkomsten uit nevenfuncties per partij")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

ax = partij_overzicht.set_index("partij")["totaal_inkomsten"].plot(kind="barh", figsize=(10, 7))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.set_xlabel("Totaal gemeld bedrag (EUR)")
ax.set_title("Totale inkomsten uit nevenfuncties per partij")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

data = partij_overzicht.sort_values("totaal_inkomsten", ascending=True)

ax = data.set_index("partij")["totaal_inkomsten"].plot(kind="barh", figsize=(10, 7))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.set_xlabel("Totaal gemeld bedrag (EUR)")
ax.set_ylabel("Partij")
ax.set_title("Totale inkomsten uit nevenfuncties per partij")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

data = (
    partij_overzicht
    .nlargest(15, "totaal_inkomsten")
    .sort_values("totaal_inkomsten", ascending=True)
)

ax = data.set_index("partij")["totaal_inkomsten"].plot(kind="barh", figsize=(10, 7))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.set_xlabel("Totaal gemeld bedrag (EUR)")
ax.set_ylabel("Partij")
ax.set_title("Top 15: totale inkomsten uit nevenfuncties per partij")
plt.tight_layout()
plt.show()

In [ ]:
partij_2025 = query_df("""
WITH inkomsten AS (
    SELECT
        n.persoon_id,
        SUM(
            CASE
                WHEN LOWER(i.frequentie) LIKE '%maand%' THEN i.bedrag * 12
                ELSE i.bedrag
            END
        ) AS totaal_bedrag
    FROM silver.persoon_nevenfunctie_inkomsten i
    JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
    WHERE i.bedrag > 0
      AND i.bedrag_valuta = 'EUR'
      AND i.jaar = 2023
    GROUP BY n.persoon_id
),
politici AS (
    SELECT DISTINCT zp.persoon_id, f.naam_nl AS partij
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT
    p.partij,
    COUNT(*) AS aantal_politici,
    COUNT(i.persoon_id) AS aantal_met_inkomsten,
    COALESCE(SUM(i.totaal_bedrag), 0) AS totaal_inkomsten
FROM politici p
LEFT JOIN inkomsten i ON i.persoon_id = p.persoon_id
GROUP BY p.partij
ORDER BY totaal_inkomsten DESC
""")
partij_2025

In [ ]:
wisselaars_aantal = query_df("""
WITH politici AS (
    SELECT DISTINCT zp.persoon_id, f.id AS fractie_id
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT COUNT(*) AS aantal_wisselaars
FROM (
    SELECT persoon_id
    FROM politici
    GROUP BY persoon_id
    HAVING COUNT(DISTINCT fractie_id) > 1
)
""")
wisselaars_aantal

In [ ]:
wisselaars = query_df("""
WITH politici AS (
    SELECT DISTINCT zp.persoon_id, f.id AS fractie_id, f.naam_nl AS partij
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT
    pe.voornamen,
    pe.achternaam,
    COUNT(DISTINCT po.fractie_id) AS aantal_partijen,
    STRING_AGG(DISTINCT po.partij, ', ') AS partijen
FROM politici po
JOIN silver.persoon pe ON pe.id = po.persoon_id
GROUP BY pe.id, pe.voornamen, pe.achternaam
HAVING COUNT(DISTINCT po.fractie_id) > 1
ORDER BY aantal_partijen DESC, pe.achternaam
""")
wisselaars

In [ ]:
"""
SELECT
    COUNT(*) AS aantal_politici,
FROM politici p
LEFT JOIN inkomsten i ON i.persoon_id = p.persoon_id
GROUP BY p.partij
ORDER BY totaal_inkomsten DESC

"""

In [ ]:
wisselaars_aantal = query_df("""
WITH politici AS (
    SELECT DISTINCT
        zp.persoon_id,
        CASE
            WHEN LOWER(f.naam_nl) LIKE '%groenlinks%'
              OR LOWER(f.naam_nl) LIKE '%progressief nederland%'
            THEN 'GroenLinks / Progressief Nederland'
            ELSE f.naam_nl
        END AS partijgroep
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT COUNT(*) AS aantal_wisselaars
FROM (
    SELECT persoon_id
    FROM politici
    GROUP BY persoon_id
    HAVING COUNT(DISTINCT partijgroep) > 1
)
""")
wisselaars_aantal

In [ ]:
wisselaars = query_df("""
WITH politici AS (
    SELECT DISTINCT
        zp.persoon_id,
        CASE
            WHEN LOWER(f.naam_nl) LIKE '%groenlinks%'
              OR LOWER(f.naam_nl) LIKE '%progressief nederland%'
            THEN 'GroenLinks / Progressief Nederland'
            ELSE f.naam_nl
        END AS partijgroep
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
)
SELECT
   pe.voornamen,
    pe.achternaam,
    COUNT(DISTINCT po.partijgroep) AS aantal_partijen,
    STRING_AGG(DISTINCT po.partijgroep, ', ') AS partijen
FROM politici po
JOIN silver.persoon pe ON pe.id = po.persoon_id
GROUP BY pe.id, pe.voornamen, pe.achternaam
HAVING COUNT(DISTINCT po.partijgroep) > 1
ORDER BY aantal_partijen DESC, pe.achternaam
""")
wisselaars.head(50)

In [ ]:
wisselaars_inkomsten = query_df("""
WITH politici AS (
    SELECT DISTINCT
        zp.persoon_id,
        CASE
            WHEN LOWER(f.naam_nl) LIKE '%groenlinks%'
              OR LOWER(f.naam_nl) LIKE '%progressief nederland%'
            THEN 'GroenLinks / Progressief Nederland'
            ELSE f.naam_nl
        END AS partijgroep
    FROM silver.fractie_zetel_persoon zp
    JOIN silver.fractie_zetel z ON z.id = zp.fractie_zetel_id
    JOIN silver.fractie f ON f.id = z.fractie_id
),
wisselaars AS (
    SELECT
        persoon_id,
        COUNT(DISTINCT partijgroep) AS aantal_partijen,
        STRING_AGG(DISTINCT partijgroep, ', ') AS partijen
    FROM politici
    GROUP BY persoon_id
    HAVING COUNT(DISTINCT partijgroep) > 1
),
inkomsten AS (
    SELECT
        n.persoon_id,
        COUNT(DISTINCT n.id) AS aantal_nevenfuncties_met_inkomsten,
        SUM(
            CASE
                WHEN LOWER(i.frequentie) LIKE '%maand%' THEN i.bedrag * 12
                ELSE i.bedrag
            END
        ) AS totaal_bedrag
    FROM silver.persoon_nevenfunctie_inkomsten i
    JOIN silver.persoon_nevenfunctie n ON n.id = i.persoon_nevenfunctie_id
    WHERE i.bedrag > 0
      AND i.bedrag_valuta = 'EUR'
    GROUP BY n.persoon_id
)
SELECT
    pe.voornamen,
    pe.achternaam,
    w.partijen,
    i.aantal_nevenfuncties_met_inkomsten,
    i.totaal_bedrag
FROM wisselaars w
JOIN inkomsten i ON i.persoon_id = w.persoon_id
JOIN silver.persoon pe ON pe.id = w.persoon_id
ORDER BY i.totaal_bedrag DESC
""")
wisselaars_inkomsten